Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\1pasos_mlp_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 7)
Dimensiones de Y: (43788, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43788, 84)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 84)
Las dimensiones de testX son:  (8801, 84)
Las dimensiones de valX son:  (4336, 84)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

49/49 - 7s - 144ms/step - ia: 0.3300 - loss: 1.3721 - mae: 0.8087 - rmse: 1.1561 - smape: 1.2994 - val_ia: 0.2886 - val_loss: 0.5387 - val_mae: 0.5188 - val_rmse: 0.6589 - val_smape: 1.1262

Epoch 2/128                                           

49/49 - 1s - 12ms/step - ia: 0.2942 - loss: 1.3247 - mae: 0.8080 - rmse: 1.1388 - smape: 1.3662 - val_ia: 0.2799 - val_loss: 0.5266 - val_mae: 0.5305 - val_rmse: 0.6631 - val_smape: 1.2587

Epoch 3/128                                           

49/49 - 0s - 9ms/step - ia: 0.2710 - loss: 1.2824 - mae: 0.8075 - rmse: 1.1261 - smape: 1.4163 - val_ia: 0.2691 - val_loss: 0.5269 - val_mae: 0.5437 - val_rmse: 0.6718 - val_smape: 1.3951

Epoch 4/128                                           

49/49 - 0s - 5ms/step - ia: 0.2457 - loss: 1.2672 - mae: 0.8192 - rmse: 1.1155 - smape: 1.4631 - val_ia: 0.2693 - val_loss: 0.5310 - val_mae: 0.5549 - val_rmse: 0.6801 - val_smape: 1.5221

Epoch 5/128      

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



385/385 - 4s - 11ms/step - ia: 0.8362 - loss: 0.1478 - mae: 0.2369 - rmse: 0.3252 - smape: 0.4964 - val_ia: 0.5875 - val_loss: 0.0720 - val_mae: 0.1649 - val_rmse: 0.2154 - val_smape: 0.4891

Epoch 2/128                                                                      

385/385 - 2s - 6ms/step - ia: 0.8698 - loss: 0.0789 - mae: 0.1877 - rmse: 0.2592 - smape: 0.4238 - val_ia: 0.5713 - val_loss: 0.0703 - val_mae: 0.1660 - val_rmse: 0.2155 - val_smape: 0.4939

Epoch 3/128                                                                      

385/385 - 1s - 3ms/step - ia: 0.8747 - loss: 0.0729 - mae: 0.1777 - rmse: 0.2497 - smape: 0.4064 - val_ia: 0.5322 - val_loss: 0.0901 - val_mae: 0.2014 - val_rmse: 0.2539 - val_smape: 0.5494

Epoch 4/128                                                                      

385/385 - 1s - 4ms/step - ia: 0.8775 - loss: 0.0715 - mae: 0.1751 - rmse: 0.2467 - smape: 0.4082 - val_ia: 0.5750 - val_loss: 0.0771 - val_mae: 0.1698 - val_rmse: 0.2213 - val_s

...

385/385 - 3s - 9ms/step - ia: 0.3641 - loss: 1.2028 - mae: 0.8191 - rmse: 1.0610 - smape: 1.3231 - val_ia: 0.2883 - val_loss: 0.2991 - val_mae: 0.3930 - val_rmse: 0.4503 - val_smape: 1.0193

Epoch 2/128                                                                       

385/385 - 2s - 6ms/step - ia: 0.5201 - loss: 0.7727 - mae: 0.6608 - rmse: 0.8538 - smape: 1.0946 - val_ia: 0.3344 - val_loss: 0.2059 - val_mae: 0.3293 - val_rmse: 0.3883 - val_smape: 0.8780

Epoch 3/128                                                                       

385/385 - 1s - 2ms/step - ia: 0.5847 - loss: 0.6147 - mae: 0.5957 - rmse: 0.7615 - smape: 0.9970 - val_ia: 0.3683 - val_loss: 0.1714 - val_mae: 0.2942 - val_rmse: 0.3527 - val_smape: 0.7902

Epoch 4/128                                                                       

385/385 - 1s - 3ms/step - ia: 0.6307 - loss: 0.5050 - mae: 0.5338 - rmse: 0.6910 - smape: 0.9122 - val_ia: 0.3946 - val_loss: 0.1487 - val_mae: 0.2680 - val_rmse: 0.3249 - val

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



770/770 - 5s - 6ms/step - ia: 0.2145 - loss: 10.0788 - mae: 2.1300 - rmse: 2.9282 - smape: 1.5569 - val_ia: 0.1056 - val_loss: 3.4155 - val_mae: 1.5384 - val_rmse: 1.6241 - val_smape: 1.3754

Epoch 2/128                                                                        

770/770 - 2s - 2ms/step - ia: 0.2088 - loss: 9.9218 - mae: 2.1368 - rmse: 2.9233 - smape: 1.5700 - val_ia: 0.1069 - val_loss: 3.2903 - val_mae: 1.5075 - val_rmse: 1.5933 - val_smape: 1.3695

Epoch 3/128                                                                        

770/770 - 2s - 2ms/step - ia: 0.2125 - loss: 9.4301 - mae: 2.0796 - rmse: 2.8432 - smape: 1.5810 - val_ia: 0.1083 - val_loss: 3.1754 - val_mae: 1.4785 - val_rmse: 1.5646 - val_smape: 1.3639

Epoch 4/128                                                                        

770/770 - 2s - 2ms/step - ia: 0.2170 - loss: 8.8223 - mae: 2.0190 - rmse: 2.7440 - smape: 1.5531 - val_ia: 0.1097 - val_loss: 3.0687 - val_mae: 1.4511 - val_rmse: 1.5373 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



49/49 - 2s - 38ms/step - ia: 0.4917 - loss: 0.8243 - mae: 0.6504 - rmse: 0.8919 - smape: 1.1259 - val_ia: 0.5221 - val_loss: 0.2432 - val_mae: 0.3624 - val_rmse: 0.4668 - val_smape: 0.9381

Epoch 2/128                                                                         

49/49 - 0s - 7ms/step - ia: 0.6540 - loss: 0.4628 - mae: 0.5031 - rmse: 0.6777 - smape: 0.8998 - val_ia: 0.6298 - val_loss: 0.1738 - val_mae: 0.2977 - val_rmse: 0.3947 - val_smape: 0.7837

Epoch 3/128                                                                         

49/49 - 0s - 4ms/step - ia: 0.6970 - loss: 0.3796 - mae: 0.4560 - rmse: 0.6142 - smape: 0.8311 - val_ia: 0.6695 - val_loss: 0.1453 - val_mae: 0.2689 - val_rmse: 0.3610 - val_smape: 0.7217

Epoch 4/128                                                                         

49/49 - 0s - 6ms/step - ia: 0.7168 - loss: 0.3331 - mae: 0.4306 - rmse: 0.5754 - smape: 0.8007 - val_ia: 0.6937 - val_loss: 0.1276 - val_mae: 0.2504 - val_rmse: 0.3381 - val_

In [16]:
print(best)

{'activation': 3, 'batch': 4, 'dropout': 0.4, 'layers': 1.0, 'learning_rate': 0.009726326397617627, 'units': 3}


In [17]:
#{'activation': 1, 'batch': 2, 'dropout': 0.2, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}